In [2]:
!pip install google-adk google-generativeai -q

import os
import re
import asyncio
from IPython.display import display, Markdown
from google.adk.agents import Agent, SequentialAgent, LoopAgent, ParallelAgent
from google.adk.tools import google_search, ToolContext
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService, Session
from google.genai.types import Content, Part
from getpass import getpass

# Secure API Configuration
api_key = getpass('Enter your Google API Key: ')
os.environ['GOOGLE_API_KEY'] = api_key

# Initialize Session Management
session_service = InMemorySessionService()
my_user_id = "smoke_free_man_001"

print("✅ Environment ready!")

Enter your Google API Key: ··········
✅ Environment ready!


In [4]:
async def run_agent_query(agent, query, session, user_id, is_router=False):
    """Executes a query for a given agent and prints the event stream."""
    if not is_router:
        print(f"\n🚀 {agent.name.upper()} is thinking...")

    runner = Runner(agent=agent, session_service=session_service, app_name=agent.name)
    final_response = ""

    try:
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session.id,
            new_message=Content(parts=[Part(text=query)], role="user")
        ):
            if event.is_final_response():
                final_response = event.content.parts[0].text
    except Exception as e:
        final_response = f"An error occurred: {e}"

    if not is_router:
        print(f"✅ Final Response from {agent.name}:")
        display(Markdown(final_response))
        print("-" * 30)

    return final_response

print("⚙️ Runner engine initialized.")

⚙️ Runner engine initialized.


In [12]:

ko_chit_tee_agent = Agent(
    name="ko_chit_tee",
    model="gemini-2.5-flash",
    instruction="""You are Ko Chit Tee, a financial expert from Myanmar.
    Assume 300 MMK per cigarette. Use the Myanmar language (Burmese).
    Always start with '[ကိုချစ်တီး] ပြန်လည်ဖြေကြားချက် -'.""",
    output_key="financial_perspective"
)

dr_yan_ku_agent = Agent(
    name="dr_yan_ku",
    model="gemini-2.5-flash", tools=[google_search],
    instruction="""You are Dr. Yan Ku, a medical expert.
    Use Myanmar language to provide health facts.
    Always start with '[ဒေါက်တာရမ်းကု မပု's ယောက်ျား] ပြန်လည်ဖြေကြားချက် -'.""",
    output_key="biological_perspective"
)

ari_agent = Agent(
    name="ari_agent",
    model="gemini-2.5-flash",
    instruction="""You are ARI, a counselor. Use Myanmar language.
    Focus on empathy and university stress.
    Always start with '[ARI] ပြန်လည်ဖြေကြားချက် -'.""",
    output_key="empathy_response"
)




craving_buster_agent = SequentialAgent(
    name="craving_buster_agent",
    sub_agents=[ari_agent.copy(), dr_yan_ku_agent.copy()],
    description="Acute cravings support flow."
)


bio_clone = dr_yan_ku_agent.copy(update={"name": "bio_parallel"})
finance_clone = ko_chit_tee_agent.copy(update={"name": "finance_parallel"})

parallel_research = ParallelAgent(
    name="parallel_research",
    sub_agents=[finance_clone, bio_clone]
)

synthesis_agent = Agent(
    name="synthesis_agent", model="gemini-2.5-flash",
    instruction="Combine financial and biological info into a Myanmar summary."
)

motivation_dashboard_agent = SequentialAgent(
    name="motivation_dashboard_agent",
    sub_agents=[parallel_research, synthesis_agent]
)

print("✅ Specialists နဲ့ Workflows အားလုံး တည်ဆောက်ပြီးပါပြီ။")

✅ Specialists နဲ့ Workflows အားလုံး တည်ဆောက်ပြီးပါပြီ။


/tmp/ipykernel_22400/1974038369.py:35: PydanticDeprecatedSince20: The `copy` method is deprecated; use `model_copy` instead. See the docstring of `BaseModel.copy` for details about how to handle `include` and `exclude`. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  sub_agents=[ari_agent.copy(), dr_yan_ku_agent.copy()],
/tmp/ipykernel_22400/1974038369.py:40: PydanticDeprecatedSince20: The `copy` method is deprecated; use `model_copy` instead. See the docstring of `BaseModel.copy` for details about how to handle `include` and `exclude`. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  bio_clone = dr_yan_ku_agent.copy(update={"name": "bio_parallel"})
/tmp/ipykernel_22400/1974038369.py:41: PydanticDeprecatedSince20: The `copy` method is deprecated; use `model_copy` instead. See the docstring of `BaseModel.copy` for details 

In [13]:

router_agent = Agent(
    name="router_agent",
    model="gemini-2.5-flash",
    instruction="""Analyze the user query. Return only the name of the best agent:
    - 'ko_chit_tee': Only money.
    - 'dr_yan_ku': Only health.
    - 'ari_agent': Only stress.
    - 'craving_buster_agent': Immediate cravings.
    - 'motivation_dashboard_agent': General benefits of quitting."""
)

worker_agents = {
    "ko_chit_tee": ko_chit_tee_agent,
    "dr_yan_ku": dr_yan_ku_agent,
    "ari_agent": ari_agent,
    "craving_buster_agent": craving_buster_agent,
    "motivation_dashboard_agent": motivation_dashboard_agent
}

print("✅ Router နဲ့ Worker Dictionary အဆင်သင့်ဖြစ်ပါပြီ။")

✅ Router နဲ့ Worker Dictionary အဆင်သင့်ဖြစ်ပါပြီ။


In [ ]:
async def main():
    print("🚬 ဆေးလိပ် ဖြတ်ချင်နေပီလား? Lungthy Agent ရှိပါတယ် [စတင်နေပါပြီ]")
    print("(စနစ်မှ ထွက်လိုပါက 'exit' ဟု ရိုက်ထည့်ပါ)\n")

    while True:
        query = input("မိတ်ဆွေ : ")
        if query.lower() in ['exit', 'quit', 'ထွက်မယ်']:
            print("\n👋 ကျန်းမာတဲ့ ဘဝသစ်မှာ ပျော်ရွှင်ပါစေ!")
            break

        print(f"\n{'='*40}\n🧠 [အဆင့် ၁] မိတ်ဆွေရဲ့ ရည်ရွယ်ချက်ကို ခွဲခြမ်းစိတ်ဖြာနေသည်...")

        router_session = await session_service.create_session(app_name=router_agent.name, user_id=my_user_id)
        chosen_route = await run_agent_query(router_agent, query, router_session, my_user_id, is_router=True)

        chosen_route = str(chosen_route).strip().replace("'", "").replace('"', "")

        final_route = "ari_agent"
        for key in worker_agents.keys():
            if key in chosen_route.lower():
                final_route = key
                break

        print(f"🚀 [အဆင့် ၂] {final_route} နှင့် ချိတ်ဆက်ပေးနေသည်...")

        agent = worker_agents[final_route]
        worker_session = await session_service.create_session(app_name=agent.name, user_id=my_user_id)

        await run_agent_query(agent, query, worker_session, my_user_id)
        print(f"{'='*40}\n")

await main()

🚬 ဆေးလိပ် ဖြတ်ချင်နေပီလား? Lungthy Agent ရှိပါတယ် [စတင်နေပါပြီ]
(စနစ်မှ ထွက်လိုပါက 'exit' ဟု ရိုက်ထည့်ပါ)


🧠 [အဆင့် ၁] မိတ်ဆွေရဲ့ ရည်ရွယ်ချက်ကို ခွဲခြမ်းစိတ်ဖြာနေသည်...
🚀 [အဆင့် ၂] motivation_dashboard_agent နှင့် ချိတ်ဆက်ပေးနေသည်...

🚀 MOTIVATION_DASHBOARD_AGENT is thinking...
✅ Final Response from motivation_dashboard_agent:


ဆေးလိပ်ဖြတ်ဖို့ ကြိုးစားနေတာဟာ တကယ်ကို အံ့ဩစရာကောင်းပြီး လေးစားစရာ ဆုံးဖြတ်ချက်တစ်ခုပါပဲ။ သင့်ရဲ့ ကျန်းမာရေးအတွက်ရော၊ ငွေကြေးအတွက်ပါ အကောင်းဆုံး ရွေးချယ်မှုဖြစ်ပါတယ်။

**ငွေကြေးဆိုင်ရာ အကျိုးကျေးဇူးများ**

ဆေးလိပ်ဖြတ်လိုက်ခြင်းအားဖြင့် ငွေကြေးကို သိသိသာသာ စုဆောင်းနိုင်ပါတယ်။ ဆေးလိပ်တစ်လိပ်ကို ၃၀၀ ကျပ်နှုန်းနဲ့ တွက်မယ်ဆိုရင်:

*   **တစ်ရက်ကို ဆေးလိပ် ၁၀ လိပ် သောက်တယ်ဆိုရင်:**
    *   တစ်လကို **၉၀,၀၀၀ ကျပ်** စုဆောင်းမိမှာဖြစ်ပြီး၊ တစ်နှစ်ကို **၁,၀၉၅,၀၀၀ ကျပ်** (ဆယ်သိန်းကျော်) စုမိမှာပါ။
*   **တစ်ရက်ကို ဆေးလိပ် ၂၀ လိပ် (တစ်ဘူး) သောက်တယ်ဆိုရင်:**
    *   တစ်လကို **၁၈၀,၀00 ကျပ်** စုဆောင်းမိမှာဖြစ်ပြီး၊ တစ်နှစ်ကို **၂,၁၉၀,၀၀၀ ကျပ်** (နှစ်ဆယ်သိန်းကျော်) အထိ စုမိပါလိမ့်မယ်။

ဒီလိုစုဆောင်းမိတဲ့ငွေတွေကို ကိုယ်နှစ်သက်ရာ ဝယ်ယူတာ၊ ရင်းနှီးမြှုပ်နှံတာ၊ မိသားစုနဲ့အတူ ခရီးထွက်တာ၊ အရေးပေါ်ရန်ပုံငွေအဖြစ်ထားတာ ဒါမှမဟုတ် ကိုယ်ပိုင်စီးပွားရေးတစ်ခု စတင်ဖို့အတွက် အရင်းအနှီးအဖြစ် အသုံးချနိုင်ပါတယ်။

**ကျန်းမာရေးဆိုင်ရာ အကျိုးကျေးဇူးများ**

ဆေးလိပ်ဖြတ်လိုက်တာနဲ့ သင့်ခန္ဓာကိုယ်ဟာ ချက်ချင်းဆိုသလို ကောင်းကျိုးတွေ ခံစားရပါလိမ့်မယ်။

*   **မိနစ်ပိုင်း/နာရီပိုင်းအတွင်း** သွေးခုန်နှုန်း၊ သွေးပေါင်ချိန်နဲ့ သွေးတွင်းအောက်ဆီဂျင် ပုံမှန်ပြန်ဖြစ်လာပြီး၊ နီကိုတင်းနဲ့ ကာဗွန်မိုနောက်ဆိုဒ်ဓာတ်ပမာဏတွေ လျော့ကျသွားပါလိမ့်မယ်။
*   **ရက်ပိုင်းအတွင်း** နှလုံးရောဂါဖြစ်နိုင်ခြေ တစ်ဝက်ခန့် လျော့ကျပြီး၊ အရသာခံနိုင်စွမ်း၊ အနံ့ခံနိုင်စွမ်း ပိုကောင်းလာကာ အသက်ရှူရတာလည်း ပိုမိုကောင်းမွန်လာပါတယ်။
*   **လပိုင်း/နှစ်ပိုင်းအတွင်း** အဆုတ်တွေ ပိုသန့်ရှင်းပြီး အားကောင်းလာကာ သွေးစီးဆင်းမှု ပိုကောင်းလာပါတယ်။ နှလုံးသွေးကြောကျဉ်းရောဂါ၊ လေဖြတ်နိုင်ခြေ၊ သားအိမ်ခေါင်းကင်ဆာနဲ့ အဆုတ်ကင်ဆာ ဖြစ်နိုင်ခြေတွေ သိသိသာသာ လျော့ကျသွားမှာပါ။
*   **၁၅ နှစ်အကြာမှာ** နှလုံးရောဂါဖြစ်နိုင်ခြေဟာ ဆေးလိပ်လုံးဝမသောက်ဖူးသူနဲ့ အတူတူနီးပါး ဖြစ်လာပါလိမ့်မယ်။

ဒါ့အပြင် ခံတွင်းနံ့ကောင်းလာတာ၊ သွားတွေဖြူလာတာ၊ အဝတ်အစားနဲ့ ဆံပင်မှာ ဆေးလိပ်နံ့မနံတော့တာ၊ အစားအသောက်ပိုစားကောင်းတာနဲ့ အရေးအကြီးဆုံးကတော့ ဘေးပတ်ဝန်းကျင်က လူများရဲ့ ကျန်းမာရေးကိုပါ ကာကွယ်ပေးနိုင်တဲ့ အကျိုးကျေးဇူးတွေလည်း ရရှိမှာပါ။

**ဆေးလိပ်ဖြတ်ရန် အကြံပြုချက်များ**

ဆေးလိပ်ဖြတ်ဖို့အတွက် စနစ်တကျ ပြင်ဆင်ပြီး ကြိုးစားဖို့ လိုအပ်ပါတယ်။

1.  **ဖြတ်မည့်နေ့ကို သတ်မှတ်ပါ:** ပြက္ခဒိန်တွင် မှတ်သားထားပြီး ကြိုတင်ပြင်ဆင်ပါ။
2.  **အကြောင်းရင်းများကို ချရေးထားပါ:** ဘာကြောင့် ဆေးလိပ်ဖြတ်ချင်တာလဲဆိုတဲ့ အကြောင်းပြချက်တွေကို ချရေးပြီး မကြာခဏ ပြန်ကြည့်ပါ။
3.  **ဆေးလိပ်သောက်စေသည့်အရာများကို ရှာဖွေဖယ်ရှားပါ:** သင့်ကို ဆေးလိပ်သောက်ချင်စိတ်ဖြစ်ပေါ်စေတဲ့ လူ၊ နေရာ၊ အခြေအနေနဲ့ အရာဝတ္ထုတွေကို ရှောင်ရှားပါ။ အိမ်၊ ကား၊ ရုံးထဲက စီးကရက်၊ မီးခြစ်၊ ပြာခွက်တွေအားလုံးကို ရှင်းထုတ်ပစ်ပါ။
4.  **ပါးစပ်ထဲတစ်ခုခု ထည့်ထားပါ:** ဆေးလိပ်သောက်ချင်စိတ် ဖြစ်ပေါ်လာပါက ပီကေဝါးခြင်း၊ မြေပဲ သို့မဟုတ် နေကြာစေ့ စားခြင်း၊ ရေအေးလေး တစ်ကျိုက်စီငုံခြင်း စတာတွေ လုပ်နိုင်ပါတယ်။
5.  **တက်ကြွစွာ လှုပ်ရှားပါ:** နေ့စဉ် လေ့ကျင့်ခန်းလုပ်ခြင်း၊ လမ်းလျှောက်ထွက်ခြင်း စတာတွေက ဆေးလိပ်သောက်ချင်စိတ်ကို လျှော့ချပေးနိုင်ပါတယ်။
6.  **အပေါင်းအသင်း၏ အကူအညီရယူပါ:** ဆေးလိပ်ဖြတ်ရန် ကြိုးစားနေကြောင်း မိသားစု၊ သူငယ်ချင်းများကို ပြောပြပြီး ၎င်းတို့၏ ပံ့ပိုးကူညီမှုကို ရယူပါ။
7.  **သောက်ချင်စိတ်ကို ထိန်းချုပ်ပါ:** ဆေးလိပ်သောက်ချင်စိတ် ပြင်းပြလာပါက အများအားဖြင့် ၅-၁၀ မိနစ်ခန့်သာ ကြာမြင့်တတ်ပါတယ်။ အချိန်ဆွဲတာ၊ အသက်ပြင်းပြင်းရှူတာ၊ ရေများများသောက်တာနဲ့ အခြားအရာတစ်ခုခုလုပ်တာတွေနဲ့ ထိုအချိန်ကို ကျော်ဖြတ်ပါ။
8.  **ဆေးဝါးအကူအညီရယူပါ:** နီကိုတင်းအစားထိုးကုထုံး (NRT) များဖြစ်သည့် ကပ်ခွာ၊ ပီကေ စသည်တို့ကို အသုံးပြုနိုင်ပါတယ်။ ဆရာဝန်ညွှန်ကြားချက်ဖြင့် ဆေးဝါးများ သောက်သုံးခြင်းကလည်း ဆေးလိပ်ဖြတ်ရန် အောင်မြင်မှုနှုန်းကို နှစ်ဆတိုးစေပါတယ်။

**နီကိုတင်းပြတ်တောက်မှု လက္ခဏာများ**
ဆေးလိပ်ဖြတ်စတွင် စိတ်တိုတာ၊ ဂဏှာမငြိမ်တာ၊ အိပ်မပျော်တာ၊ ခေါင်းကိုက်တာ စတဲ့ နီကိုတင်းဓာတ်ပြတ်တောက်မှု လက္ခဏာတွေ ဖြစ်ပေါ်နိုင်ပေမယ့် ဒါတွေဟာ အန္တရာယ်မရှိဘဲ အချိန်ကြာလာသည်နှင့်အမျှ သက်သာသွားပါလိမ့်မယ်။ စိတ်ရှည်သည်းခံပြီး တစ်ခါတရံ ပြန်သောက်မိရင်တောင် စိတ်ဓာတ်မကျဘဲ ပြန်လည်ကြိုးစားဖို့ အရေးကြီးပါတယ်။

သင့်ရဲ့ ဒီခရီးစဉ်မှာ အောင်မြင်ပါစေလို့ အားပေးစကားပြောရင်း၊ ဒီလို ကောင်းမွန်တဲ့ ဆုံးဖြတ်ချက်အတွက် ဂုဏ်ယူမိပါတယ်ခင်ဗျာ။

------------------------------


🧠 [အဆင့် ၁] မိတ်ဆွေရဲ့ ရည်ရွယ်ချက်ကို ခွဲခြမ်းစိတ်ဖြာနေသည်...
🚀 [အဆင့် ၂] motivation_dashboard_agent နှင့် ချိတ်ဆက်ပေးနေသည်...

🚀 MOTIVATION_DASHBOARD_AGENT is thinking...
✅ Final Response from motivation_dashboard_agent:


မင်္ဂလာပါခင်ဗျာ/ရှင့်။ ကျန်းမာရေးဆိုင်ရာ အချက်အလက်များဖြစ်စေ၊ ငွေကြေးဆိုင်ရာ ကိစ္စရပ်များဖြစ်စေ၊ သိရှိလိုသည်များကို မေးမြန်းနိုင်ပါတယ်ရှင့်/ခင်ဗျာ။

------------------------------


🧠 [အဆင့် ၁] မိတ်ဆွေရဲ့ ရည်ရွယ်ချက်ကို ခွဲခြမ်းစိတ်ဖြာနေသည်...
🚀 [အဆင့် ၂] ko_chit_tee နှင့် ချိတ်ဆက်ပေးနေသည်...

🚀 KO_CHIT_TEE is thinking...
✅ Final Response from ko_chit_tee:


...

------------------------------


🧠 [အဆင့် ၁] မိတ်ဆွေရဲ့ ရည်ရွယ်ချက်ကို ခွဲခြမ်းစိတ်ဖြာနေသည်...
🚀 [အဆင့် ၂] ari_agent နှင့် ချိတ်ဆက်ပေးနေသည်...

🚀 ARI_AGENT is thinking...
✅ Final Response from ari_agent:


An error occurred: 
On how to mitigate this issue, please refer to:

https://google.github.io/adk-docs/agents/models/google-gemini/#error-code-429-resource_exhausted


429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 18.293899565s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash'}, 'quotaValue': '20'}]}, {'@type': 'type.googleapis.com/google.rpc.RetryInfo', 'retryDelay': '18s'}]}}

------------------------------


🧠 [အဆင့် ၁] မိတ်ဆွေရဲ့ ရည်ရွယ်ချက်ကို ခွဲခြမ်းစိတ်ဖြာနေသည်...
🚀 [အဆင့် ၂] dr_yan_ku နှင့် ချိတ်ဆက်ပေးနေသည်...

🚀 DR_YAN_KU is thinking...
✅ Final Response from dr_yan_ku:


[ဒေါက်တာရန်ကူ] ပြန်လည်ဖြေကြားချက် -

ဆေးလိပ်ဖြတ်ပြီးနောက် အဆုတ်၏ပြန်လည်ကောင်းမွန်လာမှုသည် အချိန်ကာလအလိုက် သိသာထင်ရှားသော အပြောင်းအလဲများ ရှိပါသည်။ သင်ဆေးလိပ်ဖြတ်လိုက်သည်နှင့် အဆုတ်သည် ချက်ချင်းပြန်လည်စတင်ကုစားပါသည်။

အဆုတ်ပြန်လည်ကောင်းမွန်လာမည့် အချိန်ဇယားမှာ အောက်ပါအတိုင်းဖြစ်ပါသည်-

*   **၁၂ နာရီအတွင်း** - သွေးထဲရှိ ကာဗွန်မိုနောက်ဆိုဒ် (carbon monoxide) ပမာဏ ပုံမှန်ပြန်ဖြစ်လာပြီး အောက်ဆီဂျင်စီးဆင်းမှု ပိုမိုကောင်းမွန်လာပါသည်။
*   **၁ ရက်မှ ၂ ရက်အတွင်း** - အဆုတ်အတွင်းရှိ ဆံချည်မျှင်လေးများ (cilia) ပြန်လည်အသက်ဝင်လာပြီး အဆုတ်မှ ချွဲနှင့် အညစ်အကြေးများကို ဖယ်ရှားပေးသည့် လုပ်ငန်းစဉ် စတင်ပါသည်။ ဆေးလိပ်သောက်ခြင်းကြောင့် ဤဆံချည်မျှင်လေးများ ထိခိုက်ပျက်စီး သို့မဟုတ် လေဖြတ်သကဲ့သို့ ဖြစ်နေတတ်ပါသည်။
*   **၁ ပတ်အတွင်း** - အသက်ရှူရ ပိုမိုလွယ်ကူလာသည်ကို လူအများစု သတိထားမိကြပါသည်။ လေပြွန်များရောင်ရမ်းခြင်းလည်း လျော့နည်းလာပါသည်။
*   **၂ ပတ်မှ ၃ လအတွင်း** - အဆုတ်၏ လုပ်ဆောင်ချက် (lung function) ၃၀% အထိ တိုးတက်လာနိုင်ပါသည်။ သွေးလည်ပတ်မှုလည်း ကောင်းမွန်လာပါသည်။ ဤကာလတွင် ဆံချည်မျှင်လေးများ လုံးဝနီးပါး ပြန်လည်ကောင်းမွန်လာပြီး ချွဲသလိပ်ရှင်းလင်းမှုလည်း ပိုမိုကောင်းမွန်လာပါသည်။
*   **၁ လမှ ၉ လအတွင်း** - ချောင်းဆိုးခြင်းနှင့် အသက်ရှူမဝခြင်းကဲ့သို့သော လက္ခဏာများ လျော့နည်းလာပါသည်။ အဆုတ်၏ ချွဲသလိပ်ရှင်းလင်းနိုင်စွမ်းလည်း တိုးတက်လာပြီး ရောဂါပိုးဝင်ရောက်နိုင်ခြေကို လျော့နည်းစေပါသည်။
*   **၆ လအတွင်း** - အဆုတ်သည် ပြန်လည်ကောင်းမွန်လာမှုတွင် သိသာထင်ရှားသော တိုးတက်မှုများ ရှိလာပါသည်။ ဆံချည်မျှင်များသည် ထိရောက်စွာ အလုပ်လုပ်ပြီး ချွဲသလိပ်ရှင်းလင်းမှု ပုံမှန်ဖြစ်လာကာ လေပြွန်များရောင်ရမ်းခြင်းလည်း သိသိသာသာ လျော့နည်းလာပါသည်။
*   **၁ နှစ်အတွင်း** - နှလုံးရောဂါဖြစ်နိုင်ခြေသည် ဆေးလိပ်ဆက်သောက်သူများထက် တစ်ဝက်ခန့် လျော့နည်းသွားပါသည်။ အဆုတ်၏ လုပ်ဆောင်ချက်သည်လည်း ပုံမှန်နီးပါး ပြန်ဖြစ်လာပါသည်။
*   **၅ နှစ်အတွင်း** - အဆုတ်ကင်ဆာနှင့် COPD ကဲ့သို့သော ဆေးလိပ်နှင့်ဆက်စပ်သော အဆုတ်ရောဂါများ ဖြစ်ပွားနိုင်ခြေ သိသိသာသာ လျော့နည်းလာပါသည်။ ပါးစပ်၊ လည်ချောင်း၊ အစာပြွန်နှင့် ဆီးအိတ်ကင်ဆာ ဖြစ်နိုင်ခြေလည်း တစ်ဝက်ခန့် လျော့နည်းလာပါသည်။
*   **၁၀ နှစ်အတွင်း** - အဆုတ်ကင်ဆာဖြင့် သေဆုံးနိုင်ခြေသည် ဆေးလိပ်ဆက်သောက်သူများထက် တစ်ဝက်ခန့် လျော့နည်းသွားပြီး အဆုတ်သည် ဆေးလိပ်မသောက်သူတစ်ဦး၏ အဆုတ်ကဲ့သို့ပင် လုပ်ဆောင်ချက်များ ရှိလာပါသည်။
*   **၁၅ နှစ်အတွင်း** - နှလုံးသွေးကြောဆိုင်ရာ ရောဂါဖြစ်နိုင်ခြေသည် ဆေးလိပ်မသောက်ဖူးသူတစ်ဦးနှင့် တူညီလာပါသည်။

**အရေးကြီးသောအချက်များ**

*   ဆေးလိပ်ဖြတ်ခြင်းသည် အသက်အရွယ်မရွေး အကျိုးရှိပါသည်။
*   ဆေးလိပ်ဖြတ်ပြီးနောက် အချို့သောပျက်စီးမှုများ (ဥပမာ- emphysema ကြောင့် အဆုတ်ထိခိုက်မှု) သည် အမြဲတမ်းပြန်မကောင်းနိုင်သော်လည်း ဆေးလိပ်ဖြတ်ခြင်းသည် နောက်ထပ်ပျက်စီးမှုများကို ကာကွယ်ပေးပြီး ရောဂါတိုးတက်မှုကို နှေးကွေးစေပါသည်။
*   အဆုတ်သည် မိမိကိုယ်မိမိ ပြန်လည်ကုစားနိုင်သော အင်္ဂါအစိတ်အပိုင်းတစ်ခုဖြစ်ပါသည်။

ဆေးလိပ်ဖြတ်ခြင်းသည် သင်၏အဆုတ်နှင့် ခန္ဓာကိုယ်တစ်ခုလုံးအတွက် အကောင်းဆုံးအရာတစ်ခုဖြစ်ပါသည်။

------------------------------

